# Code was run on Colab Pro

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import collections
import torch.optim as optim
from torch.optim import Optimizer
import time
import matplotlib.pyplot as plt

from AdamW          import AdamW
from utils          import utility, misreportUtility, misreportOptimization, trueUtility, loss
from networks       import AdditiveMechanism, Misreports,AllocationNet,PaymentNet
from restrictedAdam import Adam 

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.cuda.set_device(4)

# Set Random Seed 

In [3]:
# Initializing seeds
torch.manual_seed(0)
np.random.seed(0)

# Testing Function

In [4]:
def test(nBatch, nbrInit, R, gamma=0.001, minimum=0, maximum=1):
    
    """ This function computes the regret and payment of mechanism on a test set of size nBatch
        The optimal misreport is computed by optimizing the utility function (not by using the Misreport network)
        for R gradient steps (of stepsize gamma) and starting from nbrInit initialization, we only keep the best misreport
        To compute the regret we evaluate the mechanism at the misreport and compare to the valuation
        minimum and maximum indicate the range of the valuations
    """
    
    true = np.random.rand(nBatch,nAgent,nObject)

    localMisreports     = np.random.rand(nBatch,nbrInit,nAgent,nObject)
    batchMisreports     = torch.tensor(localMisreports).float().to(device)
    batchTrueValuations = torch.tensor(true).float().to(device)
    batchMisreports.requires_grad = True
    
    opt = Adam([batchMisreports], lr=gamma)
    
    for k in range(R):
        advU         = misreportUtility(mechanism,batchTrueValuations,batchMisreports)
        los          =  -1*torch.mean(advU).to(device)
        los.backward()
        opt.step(restricted= True, min=minimum, max=maximum)
        opt.zero_grad()
    
    misReportUtilityMax  = torch.max(advU, dim =1)[0]
    mechanism.zero_grad()
    allocation, payment = mechanism(batchTrueValuations)
    regret = F.relu(misReportUtilityMax -utility(batchTrueValuations, allocation, payment))
    mregret= torch.sum(torch.mean(regret, dim=0)).to(device)
    mregret= float(mregret.cpu().detach().numpy())

    with torch.no_grad():
        l,rMean,p = loss(payment, regret)

    testRegret.append(mregret)
    testPayment.append(float(p.detach().cpu().numpy() ))
    testOptimal.append(float((-l).detach().cpu().numpy())**2)
    print("Total regret: ",'{0:.5f}'.format(mregret), "Average regret per bidder: ",'{0:.5f}'.format(mregret/nAgent), " Optimal Revenue: ",'{0:.3f}'.format(float((-l).detach().cpu().numpy())**2), " payment: ",'{0:.3f}'.format(float(p.detach().cpu().numpy() )))

# Initializing Networks

In [5]:
nAgent   = 2
nObject  = 2

# Parameters for the mechanism (payment and allocation network)
nLayersAllocation   = 5
nLayersPayment      = 5
widthAllocation     = 200
widthPayment        = 200

# Parameters for the misreport network
nLayersMisreport    = 5
widthMisreport      = 200

gamma              = 0.001 
testBatch          = 10000

nExperiments       = 200000
batchSize          = 500
nbrBatches         = int(nExperiments/batchSize)



alloc_net = AllocationNet(nAgent, nObject, nLayersAllocation, widthAllocation).to(device)
opt_alloc = AdamW(alloc_net.parameters(), lr=0.0005)

pay_net   = PaymentNet(nAgent, nObject, nLayersAllocation, widthAllocation).to(device)
opt_pay   = AdamW(pay_net.parameters(),   lr=0.0005)

mechanism            = AdditiveMechanism(nAgent, nObject, nLayersAllocation, widthAllocation).to(device)
optimizerMechanism   = AdamW(mechanism.parameters(), lr=0.0005)

misreport            = Misreports(nAgent,nObject,nLayersMisreport, widthMisreport).to(device)
optimizerMisreport   = AdamW(misreport.parameters(), lr=0.0005)

In [6]:
testRegret    = []
testMaxRegret = []
testPayment   = []
testOptimal   = []
testTime      = []
testIteration = [0]

# range of valuations
minimum            = 0
maximum            = 1

In [7]:
@torch.no_grad()
def myerson_itemwise_allocation_payment(values, reserve=0.5):
    """
    values:  (B, A, O)  估值矩阵
    reserve: 保留价
    return:  alloc (B, A, O), pay_myr (B, A)
    """
    B, A, O = values.shape
    # 逐物品取 top-2 出价
    top2 = values.topk(k=2, dim=1)
    v1, idx1 = top2.values[:, 0, :], top2.indices[:, 0, :]  # 最高价及其索引
    v2 = top2.values[:, 1, :]                               # 第二高价

    # 判断是否超过保留价
    win = (v1 >= reserve).float()        # (B,O)
    price = torch.maximum(v2, torch.full_like(v2, reserve)) * win

    # 构造 one-hot 分配矩阵
    alloc = torch.zeros(B, A, O, device=values.device)
    alloc.scatter_(1, idx1.unsqueeze(1), win.unsqueeze(1))

    # 计算每个代理的总支付
    pay_myr = alloc * price.unsqueeze(1)  # (B,A)
    return alloc, pay_myr

# Training

In [8]:
R=10000
reserve=0.5
print("Train AllocationNet with Myerson supervision")
for t in range(1,120*nbrBatches+1):
    # 随机估值
    values = torch.rand(batchSize, nAgent, nObject, device=device)

    # 计算 Myerson 标签
    with torch.no_grad():
        alloc_myr, pay_myr = myerson_itemwise_allocation_payment(values, reserve=reserve)

    # 网络输出
    alloc_pred = alloc_net(values)

    # 监督损失
    loss_alloc = F.mse_loss(alloc_pred, alloc_myr)

    opt_alloc.zero_grad()
    loss_alloc.backward()
    opt_alloc.step()
    if t % (2*nbrBatches)==0 :
        print(f"loss={loss_alloc.item():.6f}")

Train AllocationNet with Myerson supervision
loss=0.006501
loss=0.002778
loss=0.001697
loss=0.002461
loss=0.004268
loss=0.002123
loss=0.004162
loss=0.001235
loss=0.002252
loss=0.002443
loss=0.001203
loss=0.002550
loss=0.001023
loss=0.001550
loss=0.002564
loss=0.001869
loss=0.000621
loss=0.001307
loss=0.002736
loss=0.001329
loss=0.003695
loss=0.001736
loss=0.002036
loss=0.001567
loss=0.001758
loss=0.001169
loss=0.003358
loss=0.002707
loss=0.002583
loss=0.001246
loss=0.000408
loss=0.002551
loss=0.003693
loss=0.003200
loss=0.002000
loss=0.000436
loss=0.001824
loss=0.001896
loss=0.001107
loss=0.002105
loss=0.001802
loss=0.001224
loss=0.000557
loss=0.001196
loss=0.001220
loss=0.002544
loss=0.000737
loss=0.000089
loss=0.000996
loss=0.002406
loss=0.001715
loss=0.002667
loss=0.000563
loss=0.000869
loss=0.001007
loss=0.001160
loss=0.001659
loss=0.002056
loss=0.001075
loss=0.001125


In [9]:
print("Train PaymentNet with Myerson supervision")

for t in range(1,60*nbrBatches+1):
    values = torch.rand(batchSize, nAgent, nObject, device=device)

    with torch.no_grad():
        alloc_myr, pay_myr = myerson_itemwise_allocation_payment(values, reserve=reserve)

    payments_pred = pay_net(values, alloc_myr)

    loss_pay = F.mse_loss(payments_pred, pay_myr)

    opt_pay.zero_grad()
    loss_pay.backward()
    opt_pay.step()
    if t % (2*nbrBatches)==0 :
        print(f"loss={loss_pay.item():.6f}")

Train PaymentNet with Myerson supervision
loss=0.000064
loss=0.000030
loss=0.000017
loss=0.000015
loss=0.000008
loss=0.000006
loss=0.000006
loss=0.000006
loss=0.000003
loss=0.000002
loss=0.000003
loss=0.000002
loss=0.000002
loss=0.000002
loss=0.000001
loss=0.000002
loss=0.000001
loss=0.000004
loss=0.000001
loss=0.000002
loss=0.000000
loss=0.000003
loss=0.000001
loss=0.000002
loss=0.000001
loss=0.000001
loss=0.000001
loss=0.000001
loss=0.000000
loss=0.000000


In [10]:
duration   = 0
R          = 100

i=0
mechanism            = AdditiveMechanism(nAgent, nObject, nLayersAllocation, widthAllocation).to(device)

mechanism.alloc_net.load_state_dict(alloc_net.state_dict())
mechanism.payment_net.load_state_dict(pay_net.state_dict())
optimizerMechanism   = AdamW(mechanism.parameters(), lr=0.0005)

print("Initial Test")
test(50, nbrInit=300, R=300, gamma=0.001, minimum=0, maximum=1)

for t in range(1,60*nbrBatches+1):
    
    # Reinitialize Misreport network periodically at the beginning of training
    if (t%(2*nbrBatches) ==1):
      if   t< 20*nbrBatches+2 :
    
        misreport            = Misreports(nAgent,nObject,nLayersMisreport, widthMisreport).to(device)
        optimizerMisreport   = AdamW(misreport.parameters(), lr=0.001)

    batchTrueValuations = torch.tensor(np.random.rand(batchSize,nAgent,nObject)).float().to(device)
    
    # Optimize Misreport Network for R steps
    for k in range(R):
  
        misreports          = misreport(batchTrueValuations).unsqueeze(1)
        mUtility            = misreportUtility(mechanism,batchTrueValuations,misreports).squeeze(1)
        mLoss               = torch.sum(torch.mean(-mUtility,dim=0))

        optimizerMisreport.zero_grad()
        mLoss.backward()
        optimizerMisreport.step()

    
    # Optimize Mechanism network for one step
    misreports          = misreport(batchTrueValuations).unsqueeze(1)
    mUtility            = misreportUtility(mechanism,batchTrueValuations,misreports).squeeze(1)

    allocation, payment = mechanism(batchTrueValuations)

    regret     = F.relu(mUtility -utility(batchTrueValuations, allocation, payment))
    l,rMean,p = loss(payment, regret)
        
    optimizerMechanism.zero_grad()

    l.backward()

    optimizerMechanism.step()
    
    # Test mechanism periodically
    if t % (2*nbrBatches)==0 :
        print("Batch: ", 2*int(t/(2*nbrBatches)))
        testTime.append(duration)
        testIteration.append(t/nbrBatches)
        test(50, nbrInit=300, R=300, gamma=0.001, minimum=0, maximum=1)

Initial Test


/home/wkw/ysy/women (1)/restrictedAdam.py:103: UserWarning: This overload of add_ is deprecated:
	add_(Number alpha, Tensor other)
Consider using one of the following signatures instead:
	add_(Tensor other, *, Number alpha) (Triggered internally at  ../torch/csrc/utils/python_arg_parser.cpp:1050.)
  exp_avg.mul_(beta1).add_(1 - beta1, grad)


Total regret:  0.00323 Average regret per bidder:  0.00161  Optimal Revenue:  0.736  payment:  0.843
Batch:  2
Total regret:  0.00196 Average regret per bidder:  0.00098  Optimal Revenue:  0.774  payment:  0.858
Batch:  4
Total regret:  0.00255 Average regret per bidder:  0.00127  Optimal Revenue:  0.829  payment:  0.928
Batch:  6
Total regret:  0.00147 Average regret per bidder:  0.00074  Optimal Revenue:  0.850  payment:  0.925
Batch:  8
Total regret:  0.00191 Average regret per bidder:  0.00095  Optimal Revenue:  0.878  payment:  0.966
Batch:  10
Total regret:  0.00144 Average regret per bidder:  0.00072  Optimal Revenue:  0.824  payment:  0.897
Batch:  12
Total regret:  0.00178 Average regret per bidder:  0.00089  Optimal Revenue:  0.851  payment:  0.935
Batch:  14
Total regret:  0.00141 Average regret per bidder:  0.00070  Optimal Revenue:  0.786  payment:  0.856
Batch:  16
Total regret:  0.00125 Average regret per bidder:  0.00063  Optimal Revenue:  0.798  payment:  0.864
Batch: 

# Testing

In [11]:
for i in range(200):
    test(50, nbrInit=300, R=300, gamma=0.001, minimum=0, maximum=1)

Total regret:  0.00079 Average regret per bidder:  0.00039  Optimal Revenue:  0.772  payment:  0.823
Total regret:  0.00111 Average regret per bidder:  0.00055  Optimal Revenue:  0.847  payment:  0.911
Total regret:  0.00103 Average regret per bidder:  0.00051  Optimal Revenue:  0.812  payment:  0.873
Total regret:  0.00126 Average regret per bidder:  0.00063  Optimal Revenue:  0.825  payment:  0.893
Total regret:  0.00138 Average regret per bidder:  0.00069  Optimal Revenue:  0.888  payment:  0.962
Total regret:  0.00119 Average regret per bidder:  0.00059  Optimal Revenue:  0.810  payment:  0.876
Total regret:  0.00115 Average regret per bidder:  0.00057  Optimal Revenue:  0.884  payment:  0.951
Total regret:  0.00135 Average regret per bidder:  0.00067  Optimal Revenue:  0.861  payment:  0.934
Total regret:  0.00116 Average regret per bidder:  0.00058  Optimal Revenue:  0.891  payment:  0.959
Total regret:  0.00110 Average regret per bidder:  0.00055  Optimal Revenue:  0.841  paymen

Total regret:  0.00126 Average regret per bidder:  0.00063  Optimal Revenue:  0.790  payment:  0.857
Total regret:  0.00119 Average regret per bidder:  0.00060  Optimal Revenue:  0.862  payment:  0.930
Total regret:  0.00109 Average regret per bidder:  0.00054  Optimal Revenue:  0.808  payment:  0.870
Total regret:  0.00118 Average regret per bidder:  0.00059  Optimal Revenue:  0.824  payment:  0.890
Total regret:  0.00129 Average regret per bidder:  0.00065  Optimal Revenue:  0.823  payment:  0.892
Total regret:  0.00111 Average regret per bidder:  0.00055  Optimal Revenue:  0.887  payment:  0.953
Total regret:  0.00119 Average regret per bidder:  0.00060  Optimal Revenue:  0.874  payment:  0.942
Total regret:  0.00138 Average regret per bidder:  0.00069  Optimal Revenue:  0.866  payment:  0.940
Total regret:  0.00124 Average regret per bidder:  0.00062  Optimal Revenue:  0.800  payment:  0.866
Total regret:  0.00114 Average regret per bidder:  0.00057  Optimal Revenue:  0.801  paymen

Total regret:  0.00104 Average regret per bidder:  0.00052  Optimal Revenue:  0.908  payment:  0.972
Total regret:  0.00110 Average regret per bidder:  0.00055  Optimal Revenue:  0.774  payment:  0.836
Total regret:  0.00109 Average regret per bidder:  0.00055  Optimal Revenue:  0.764  payment:  0.825
Total regret:  0.00125 Average regret per bidder:  0.00063  Optimal Revenue:  0.766  payment:  0.832
Total regret:  0.00125 Average regret per bidder:  0.00063  Optimal Revenue:  0.857  payment:  0.926
Total regret:  0.00110 Average regret per bidder:  0.00055  Optimal Revenue:  0.771  payment:  0.832
Total regret:  0.00088 Average regret per bidder:  0.00044  Optimal Revenue:  0.825  payment:  0.881
Total regret:  0.00111 Average regret per bidder:  0.00056  Optimal Revenue:  0.806  payment:  0.869
Total regret:  0.00109 Average regret per bidder:  0.00055  Optimal Revenue:  0.797  payment:  0.859
Total regret:  0.00099 Average regret per bidder:  0.00049  Optimal Revenue:  0.743  paymen

In [12]:
totalregret = np.mean(np.array(testRegret[-200:]))
revenue     = np.mean(np.array(testPayment[-200:]))
print("Final Result")
print("Total Regret = ", '{0:.5f}'.format(totalregret), "Average regret per bidder: ",'{0:.5f}'.format(totalregret/nAgent), " Optimal Revenue: ",'{0:.3f}'.format(float(np.sqrt(revenue)-np.sqrt(totalregret))**2), " payment: ",'{0:.3f}'.format(revenue))

Final Result
Total Regret =  0.00118 Average regret per bidder:  0.00059  Optimal Revenue:  0.820  payment:  0.883


In [13]:
stdregret = np.std(np.array(testRegret[-200:]))
stdrevenue= np.std(np.array(testPayment[-200:]))
print("std Regret = ", '{0:.5f}'.format(stdregret), "std regret per bidder: ",'{0:.5f}'.format(stdregret/nAgent), " std payment: ",'{0:.3f}'.format(stdrevenue))

std Regret =  0.00016 std regret per bidder:  0.00008  std payment:  0.047


In [14]:
torch.save(mechanism,'22stage2.pt')